# Distributed ML pipeline — PySpark / Spark MLlib (Databricks Free Edition)

Replication of the household **flood / landslide risk** classifiers from
*Quiliche, Maçaira, Araujo Baião & Leiras (2025), IJDRR 127, 105593* —
rebuilt with **distributed Spark MLlib** instead of single-node `XGBoost` + `BayesSearchCV`.

**What changed vs. the paper's pandas/sklearn code, and why**

| pandas / sklearn | distributed Spark | reason |
|---|---|---|
| `pd.read_csv` | `spark.read.csv` | data stays a distributed DataFrame |
| `RobustScaler` (sklearn) | `pyspark.ml.feature.RobustScaler` | native, `withCentering=True` matches sklearn |
| feature matrix `X` | `VectorAssembler` → one `features` vector | MLlib models take a single vector column |
| `scale_pos_weight` | per-row `weightCol` | MLlib's way to do cost-sensitive learning |
| `XGBClassifier` | `GBTClassifier` | MLlib's distributed gradient boosting |
| `BayesSearchCV` (skopt) | `TrainValidationSplit` + `ParamGridBuilder` | **PySpark has no Bayesian search** — only grid/random; the tuner must match the model's family |
| GIBA / GMean / specificity | confusion matrix via Spark `groupBy` | custom imbalance metrics, computed distributed |
| SHAP | `.featureImportances` + Spark PDP | SHAP is single-node only |
| `pickle.dump` | `model.write().save(...)` | MLlib-native model format |

**Serverless (Spark Connect) gotchas handled below:** a scratch Volume path for MLlib caching,
Delta-safe column names (no spaces), and a lightweight search to stay under the 1 GB model-cache limit.


## 0. Configuration

Set your real Unity Catalog names. On Free Edition the defaults are `workspace` / `default`.
Switch `LABEL` to run floods or landslides — everything else adapts automatically.

In [ ]:
# --- edit these to match your workspace ---
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "thesis_data"

LABEL = "flood_risk"          # "flood_risk"  or  "landslide_risk"

# --- derived paths (no need to edit) ---
BASE   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA   = f"{BASE}/Databases/TrainTestSplit"     # where the 8 split CSVs live
TMP    = f"{BASE}/sparkml_tmp"                  # MLlib scratch space
MODELS = f"{BASE}/models"                       # saved models

XPREF, YPREF = ("xf", "yf") if LABEL == "flood_risk" else ("xl", "yl")
print("LABEL:", LABEL, "| reading", XPREF, "/", YPREF, "from", DATA)

## 1. SparkML scratch path  *(required on serverless)*

MLlib's tuning caches intermediate DataFrames as Delta. On serverless there is no local disk it may
use, so it requires a **UC Volume** path via the `SPARKML_TEMP_DFS_PATH` environment variable.
This must run **before any `.fit()`**. If you restart the notebook, run this cell again.

In [ ]:
import os
dbutils.fs.mkdirs(TMP)
os.environ["SPARKML_TEMP_DFS_PATH"] = TMP
print("SparkML temp path:", os.environ["SPARKML_TEMP_DFS_PATH"])

## 2. Load data and attach labels

The label CSVs (`yf_*.csv`) are a single column with **no join key** — they align to the feature rows
**by position only**. To reunite them in Spark without pandas: read each in a **single partition**
(`coalesce(1)`) so `monotonically_increasing_id()` is `0..n-1` in file order, then join on that id.
Safe here because both files were written together in identical order and the data is small.

In [ ]:
from pyspark.sql import functions as F

def read(name):
    return (spark.read.option("header", True).option("inferSchema", True)
                 .csv(f"{DATA}/{name}.csv")
                 .coalesce(1)
                 .withColumn("_rid", F.monotonically_increasing_id()))

def labeled(xname, yname):
    return read(xname).join(read(yname), "_rid").drop("_rid")

train_sdf = labeled(f"{XPREF}_train", f"{YPREF}_train")
test_sdf  = labeled(f"{XPREF}_test",  f"{YPREF}_test")

print("train rows:", train_sdf.count(), "| test rows:", test_sdf.count())
train_sdf.groupBy(LABEL).count().show()   # confirm the class imbalance

## 3. Clean column names, build features, weights, and the pipeline

The region dummy columns (from pandas `get_dummies`) contain **spaces** (`13La Libertad`, …), which
Delta rejects. Sanitize all names first, then:
- **`VectorAssembler`** packs the 112 features into one `features_raw` vector,
- **`RobustScaler`** (median-centered, IQR-scaled) matches the paper's scaling,
- a **`weight`** column (`n_neg / n_pos`) gives cost-sensitive learning (the `scale_pos_weight` analog),
- **`GBTClassifier`** is the distributed booster.

In [ ]:
import re
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, RobustScaler
from pyspark.ml.classification import GBTClassifier

# 1) Delta-safe column names
def clean_cols(df):
    for c in df.columns:
        c2 = re.sub(r'[ ,;{}()\n\t=]', '_', c)
        if c2 != c:
            df = df.withColumnRenamed(c, c2)
    return df

train_sdf = clean_cols(train_sdf)
test_sdf  = clean_cols(test_sdf)

# 2) feature list (exclude IDs + label + helper cols)
ID_COLS = ["conglome", "vivienda", "hogar", "year"]
feature_cols = [c for c in train_sdf.columns if c not in ID_COLS + [LABEL, "weight"]]
assert len(feature_cols) == 112, f"expected 112 features, got {len(feature_cols)}"

# 3) cost-sensitive weights  (scale_pos_weight = n_neg / n_pos)
counts = {r[LABEL]: r["count"] for r in train_sdf.groupBy(LABEL).count().collect()}
pos_weight = counts[0] / counts[1]
add_weight = lambda df: df.withColumn(
    "weight", F.when(F.col(LABEL) == 1, F.lit(pos_weight)).otherwise(F.lit(1.0)))
train_sdf, test_sdf = add_weight(train_sdf), add_weight(test_sdf)

# 4) pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler    = RobustScaler(inputCol="features_raw", outputCol="features",
                         withCentering=True, withScaling=True)
gbt       = GBTClassifier(labelCol=LABEL, featuresCol="features", weightCol="weight", seed=0)
pipeline  = Pipeline(stages=[assembler, scaler, gbt])

print("features:", len(feature_cols), "| pos_weight:", round(pos_weight, 2))

## 4. Hyperparameter search — `TrainValidationSplit`

**Why not `CrossValidator`?** On Spark Connect, every fitted model is held in a **1 GB per-session
server-side cache**. `CrossValidator` with a 16-combo grid × 5 folds = **80 models** overflows it.
`TrainValidationSplit` does a **single** train/validation split → **one model per combo**, ~5× fewer.
Keep the grid small and `maxIter` (tree count) modest to stay under the limit.

> If you hit `ML_CACHE_SIZE_OVERFLOW`, run `dbutils.library.restartPython()` to clear models from
> earlier runs, then re-run cells 1 → 3 → 4.

In [ ]:
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

grid = (ParamGridBuilder()
        .addGrid(gbt.maxDepth, [4, 6])
        .addGrid(gbt.maxIter,  [40, 60])     # ~ n_estimators (biggest driver of model size)
        .build())                            # 4 combos

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=grid,
    evaluator=BinaryClassificationEvaluator(labelCol=LABEL, metricName="areaUnderROC"),
    trainRatio=0.8,          # 80% train / 20% validation (single split)
    parallelism=1,           # Free Edition is small; raise on a bigger cluster
    seed=0)

tvs_model = tvs.fit(train_sdf)
best = tvs_model.bestModel

print("best params:",
      {p.name: v for p, v in best.stages[-1].extractParamMap().items()
       if p.name in ("maxDepth", "maxIter")})

## 5. Evaluate with the paper's imbalance metrics

Computed from a **distributed confusion matrix** (one Spark `groupBy`), then the scalar math on the
driver. Includes **GIBA** (α = 0.9), G-Mean, sensitivity, specificity, MCC, balanced accuracy, AUC.

In [ ]:
import math

def imbalance_metrics(model, df, label, alpha=0.9):
    pred = model.transform(df)
    r = pred.select(
        F.sum(((F.col(label)==1)&(F.col("prediction")==1)).cast("int")).alias("TP"),
        F.sum(((F.col(label)==0)&(F.col("prediction")==0)).cast("int")).alias("TN"),
        F.sum(((F.col(label)==0)&(F.col("prediction")==1)).cast("int")).alias("FP"),
        F.sum(((F.col(label)==1)&(F.col("prediction")==0)).cast("int")).alias("FN"),
    ).first()
    TP, TN, FP, FN = r["TP"], r["TN"], r["FP"], r["FN"]
    tpr = TP/(TP+FN) if TP+FN else 0.0     # sensitivity / recall
    tnr = TN/(TN+FP) if TN+FP else 0.0     # specificity
    prec = TP/(TP+FP) if TP+FP else 0.0
    gms  = math.sqrt(tpr*tnr)
    giba = (1 + alpha*(tpr - tnr)) * gms   # GIBA_alpha
    den  = math.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc  = (TP*TN - FP*FN)/den if den else 0.0
    f1   = 2*prec*tpr/(prec+tpr) if prec+tpr else 0.0
    auc  = BinaryClassificationEvaluator(labelCol=label, metricName="areaUnderROC").evaluate(pred)
    return {"MCC": mcc, "GIBA": giba, "GMean": gms, "Sensitivity": tpr,
            "Specificity": tnr, "Precision": prec, "F1": f1,
            "BalancedAcc": (tpr+tnr)/2, "AUC": auc,
            "TP": TP, "TN": TN, "FP": FP, "FN": FN}

print(f"=== Test-set metrics ({LABEL}) ===")
for k, v in imbalance_metrics(best, test_sdf, LABEL).items():
    print(f"{k:>12}: {round(v,4) if isinstance(v,float) else v}")

## 6. Interpretability (distributed)

`featureImportances` replaces SHAP's mean-|value| ranking. For a **partial-dependence-style** view that
stays in Spark, bin a feature and average the predicted positive-class probability per bin.

In [ ]:
# Top-15 feature importances
imp = best.stages[-1].featureImportances
rows = sorted(((feature_cols[i], float(imp[i])) for i in range(len(feature_cols))),
              key=lambda t: -t[1])[:15]
print("Top 15 features:")
spark.createDataFrame(rows, ["feature", "importance"]).show(truncate=False)

# Distributed PDP for altitude (edit the feature / bin width as you like)
from pyspark.ml.functions import vector_to_array
pred = best.transform(test_sdf).withColumn("p", vector_to_array("probability")[1])
print("Mean predicted risk by altitude band:")
(pred.withColumn("altitude_bin", (F.col("altitude")/500).cast("int")*500)
     .groupBy("altitude_bin").agg(F.avg("p").alias("avg_risk"), F.count("*").alias("n"))
     .orderBy("altitude_bin").show())

## 7. Save the model (MLlib-native)

In [ ]:
model_path = f"{MODELS}/gbt_{LABEL}"
best.write().overwrite().save(model_path)
print("saved to:", model_path)

# reload later with:
# from pyspark.ml import PipelineModel
# reloaded = PipelineModel.load(model_path)

## Appendix — alternatives

**Want k-fold CV instead?** Swap cell 4 for `CrossValidator(..., numFolds=5)` — but only on a real
cluster; it trains `numFolds ×` more models and will overflow Free Edition's 1 GB model cache.

**Want a true Bayesian search?** PySpark has none built in. Use **Optuna** on the driver, looping over
`pipeline.fit()` and reading the AUC each trial. (Skip `hyperopt.SparkTrials` — it needs a classic cluster.)

**Want actual XGBoost on Spark?** `from xgboost.spark import SparkXGBClassifier` — a drop-in for the last
pipeline stage (`features_col`, `label_col`, `weight_col`, `num_workers=...`). It targets **classic
clusters**; on Free Edition serverless it usually fails because Spark Connect hides the RDD / barrier-mode
APIs it relies on. If so, stay on `GBTClassifier`.

**Run landslides:** set `LABEL = "landslide_risk"` in cell 0 and re-run from the top.
